In [1]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.75, # Reduce if out of memory
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)




🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-14 10:35:15 [__init__.py:244] Automatically detected platform cuda.
ERROR 06-14 10:35:24 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-14 10:35:33 [vllm_utils.py:690] Unsloth: Patching vLLM v1 graph capture
INFO 06-14 10:35:33 [vllm_utils.py:719] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM l

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 06-14 10:35:56 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 06-14 10:35:56 [config.py:1472] Using max model len 2048
WARNING 06-14 10:35:56 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 06-14 10:35:57 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['embed_tokens', 'embedding', 'lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.6.self_attn', 'model.layers.0.self_attn', 'model.layers.35.mlp', 'model.layers.34.mlp', 'model.layers.3.self_attn', 'model.layers.5.mlp', 'model.layers.3.mlp', 'model.layers.6.mlp', 'mode

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

Capturing CUDA graph shapes:   0%|          | 0/7 [00:00<?, ?it/s]

INFO 06-14 10:36:30 [model_runner.py:1671] Graph capturing finished in 4 secs, took 0.18 GiB
INFO 06-14 10:36:30 [vllm_utils.py:731] Unsloth: Patched vLLM v0 graph capture finished in 4 secs.
INFO 06-14 10:36:31 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 14.62 seconds
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'attention_norm', 'k_norm', 'q_norm', 'layer_norm2', 'norm1', 'norm', 'post_feedforward_layernorm', 'post_layernorm', 'post_attention_layernorm', 'post_per_layer_input_norm', 'norm2', 'input_layernorm', 'layer_norm1', 'ffn_norm']


Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'attention_norm', 'k_norm', 'q_norm', 'layer_norm2', 'norm1', 'norm', 'post_feedforward_layernorm', 'post_layernorm', 'cross_attn_post_attention_layernorm', 'post_attention_layernorm', 'post_per_layer_input_norm', 'norm2', 'input_layernorm', 'cross_attn_input_layernorm', 'layer_norm1', 'ffn_norm']
unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [15]:
# @title
system_prompt = '''# Minesweeper AI System Prompt

You are an AI agent playing Minesweeper.
Your objective is to maximize score while completing the game without revealing a mine.

## Allowed Actions

You may send exactly one move at a time using one of these actions:

- `reveal`
- `flag`

Each move must target exactly one tile coordinate:

- `x`
- `y`

Example move payload:

```json
{
  "action": "reveal",
  "x": 3,
  "y": 4
}
```

## Rules You Must Follow

- Revealing a flagged tile is invalid
- Flagging a revealed tile is invalid
- Revealing a mine ends the game immediately
- The game is won only when all safe tiles are revealed and all mines are correctly flagged
- Hidden mine locations are not exposed while the game is in progress

## Scoring Rules

- Reveal a safe tile: `+1` for each safe tile revealed
- Correctly flag a mine: `+2`
- Incorrectly flag a safe tile: `-2`
- Reveal a mine: immediate loss
- Win the full game: `+50`

## State Interpretation

You receive game state that includes:
- `board`

Symbol meanings:

- `.` means the tile is unrevealed
- `F` means the tile is flagged
- `_` means the tile is revealed and has zero adjacent mines
- `"1"` to `"8"` mean the tile is revealed and the value is the number of adjacent mines
- `B` means a bomb tile visible after the game is lost

Example:

```text
. . . 1
_ F 1 .
_ 2 3 _
_ _ _ _
```

Interpret the array using zero-based coordinates:

- `x` is the column index
- `y` is the row index
- `board[y][x]` is the tile value

Tile state meanings:

- `hidden`: unrevealed and unflagged
- `flagged`: currently flagged as a mine candidate
- `revealed`: safely revealed
- `mine`: appears only in terminal loss state

Interpretation of `adjacent_mines`:

- If `revealed`, the value is the number of adjacent mines
- If `0`, the tile has no adjacent mines
- If hidden or flagged during active play, `adjacent_mines` may be `null`

## Strategy Guidance

- Prefer moves that are logically certain
- Use revealed numbers to infer safe tiles and mine tiles
- Flag tiles only when there is strong justification, because incorrect flags lose points
- Prefer guaranteed safe reveals over speculative flags when uncertainty is high
- Use the safe first move to open information quickly
- Track local constraints around numbered tiles
- Avoid random reveals unless no deterministic move exists
- If forced to guess, choose the move with the lowest estimated mine risk
- Output only required json

## Decision Policy

For a given board state:
1. Read the full visible board state
2. interpret `board[y][x]` using the compact symbol rules
3. Identify deterministic safe reveals
4. Identify deterministic mine flags
5. If no deterministic move exists, estimate the least risky hidden tile
6. Return exactly one move
'''



In [16]:
chat_template = '''
{%- if messages[0].role == 'system' %}
    {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
{%- endif %}

{%- for message in messages %}
    {%- if message.content is string %}
        {%- set content = message.content %}
    {%- else %}
        {%- set content = '' %}
    {%- endif %}

    {%- if message.role == "user" %}
        {{- '<|im_start|>user\n' + content + '<|im_end|>\n' }}

    {%- elif message.role == "system" and not loop.first %}
        {{- '<|im_start|>system\n' + content + '<|im_end|>\n' }}

    {%- elif message.role == "assistant" %}
        {%- set reasoning_content = '' %}

        {%- if message.reasoning_content is string %}
            {%- set reasoning_content = message.reasoning_content %}
        {%- else %}
            {%- if '</think>' in content %}
                {%- set reasoning_content = content.split('</think>')[0].rstrip('\n').split('<think>')[-1].lstrip('\n') %}
                {%- set content = content.split('</think>')[-1].lstrip('\n') %}
            {%- endif %}
        {%- endif %}

        {%- if reasoning_content %}
            {{- '<|im_start|>assistant\n<think>\n' + reasoning_content.strip('\n') + '\n</think>\n\n' + content.lstrip('\n') + '<|im_end|>\n' }}
        {%- else %}
            {{- '<|im_start|>assistant\n' + content + '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}

{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}
'''

tokenizer.chat_template = chat_template


### Load The Datasets

In [17]:
import pandas as pd
train = pd.read_csv('/content/dataset/dataset_train.csv')
test = pd.read_csv('/content/dataset/dataset_test.csv')

#### Chat Template Example

In [5]:
user = train['input'][0]
assistant = train['output'][0]

tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ],
    tokenize = False,
    add_generation_prompt = True,
)



NameError: name 'train' is not defined

### Format the dataset

In [18]:
def format_dataset(df):
    input = df['input']
    output = df['output']

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
        {"role": "assistant", "content": output},
    ]

train['Messages'] = train.apply(format_dataset, axis=1)
test['Messages'] = test.apply(format_dataset, axis=1)


In [7]:
#check the formatted messages
train['Messages'][0]

[{'role': 'system',
  'content': '\n\nYou are an AI agent playing Minesweeper through an API.\n\nYour objective is to maximize score while completing the game without revealing a mine.\n\n## Game Objective\n\n- Reveal all safe tiles\n- Correctly flag all mine tiles\n- Avoid revealing a mine\n\n## Allowed Actions\n\nYou may send exactly one move at a time using one of these actions:\n\n- `reveal`\n- `flag`\n\nEach move must target exactly one tile coordinate:\n\n- `x`\n- `y`\n\nExample move payload:\n\n```json\n{\n  "action": "reveal",\n  "x": 3,\n  "y": 4\n}\n```\n\nThe client uses compact board output by default, so assume the board you receive is a 2D array of symbols unless the caller explicitly says otherwise.\n\n## Rules You Must Follow\n\n- The first revealed tile is always safe\n- Revealing a flagged tile is invalid\n- Flagging a revealed tile is invalid\n- Revealing a mine ends the game immediately\n- The game is won only when all safe tiles are revealed and all mines are corre

### Reward Function


In [ ]:
import ast
import json
import re


HIDDEN_CELL = "."
FLAGGED_CELL = "F"
BOMB_CELL = "B"
ZERO_CELLS = {"_", "0"}
NUMBER_CELLS = {"1", "2", "3", "4", "5", "6", "7", "8"}


def _content_from_chat_item(item):
    if isinstance(item, dict):
        return item.get("content", "")
    return str(item)


def _last_message_content(messages, role=None):
    if not isinstance(messages, (list, tuple)):
        return str(messages)

    selected = None
    for message in messages:
        if not isinstance(message, dict):
            continue
        if role is None or message.get("role") == role:
            selected = message
    return _content_from_chat_item(selected) if selected is not None else str(messages)


def _extract_move_text(completion):
    if isinstance(completion, dict):
        return completion
    if isinstance(completion, (list, tuple)):
        return _last_message_content(completion, role="assistant")
    return str(completion)


def _normalize_cell(cell):
    if isinstance(cell, dict):
        state = cell.get("state")
        if state == "hidden":
            return HIDDEN_CELL
        if state == "flagged":
            return FLAGGED_CELL
        if state == "mine":
            return BOMB_CELL
        if state == "revealed":
            adjacent = cell.get("adjacent_mines")
            return "_" if adjacent in {None, 0, "0"} else str(adjacent)

    return str(cell)


def _normalize_board(board):
    if not isinstance(board, list) or not board or not all(isinstance(row, list) for row in board):
        return None

    width = len(board[0])
    if width == 0 or any(len(row) != width for row in board):
        return None

    return [[_normalize_cell(cell) for cell in row] for row in board]


def parse_move(completion):
    """Parse a model completion into a Minesweeper move dict."""
    if isinstance(completion, dict):
        move = completion
    else:
        text = _extract_move_text(completion).strip()
        match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
        if match is None:
            return None
        raw = match.group(0)
        try:
            move = json.loads(raw)
        except json.JSONDecodeError:
            try:
                move = ast.literal_eval(raw)
            except (SyntaxError, ValueError):
                return None

    if not isinstance(move, dict):
        return None

    try:
        action = str(move["action"]).lower().strip()
        x = int(move["x"])
        y = int(move["y"])
    except (KeyError, TypeError, ValueError):
        return None

    return {"action": action, "x": x, "y": y}


def parse_board(prompt_or_state):
    """Extract a compact board from a prompt, chat messages, or API state dict."""
    if isinstance(prompt_or_state, dict):
        if "board" in prompt_or_state:
            return _normalize_board(prompt_or_state["board"])
        if "content" in prompt_or_state:
            prompt_or_state = prompt_or_state["content"]
    elif isinstance(prompt_or_state, (list, tuple)):
        prompt_or_state = _last_message_content(prompt_or_state, role="user")

    text = str(prompt_or_state).strip()
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        match = re.search(r"(\[\s*\[.*\]\s*\])", text, flags=re.DOTALL)
        if match is None:
            return None
        try:
            parsed = ast.literal_eval(match.group(1))
        except (SyntaxError, ValueError):
            return None

    if isinstance(parsed, dict):
        parsed = parsed.get("board")

    return _normalize_board(parsed)


def _neighbors(board, x, y):
    height = len(board)
    width = len(board[0]) if height else 0
    for ny in range(max(0, y - 1), min(height, y + 2)):
        for nx in range(max(0, x - 1), min(width, x + 2)):
            if nx == x and ny == y:
                continue
            yield nx, ny, board[ny][nx]


def infer_deterministic_cells(board):
    """Return cells that visible number constraints prove safe or mined."""
    safe_cells = set()
    mine_cells = set()

    for y, row in enumerate(board):
        for x, cell in enumerate(row):
            if cell not in NUMBER_CELLS:
                continue

            hidden = []
            flagged = 0
            for nx, ny, neighbor in _neighbors(board, x, y):
                if neighbor == HIDDEN_CELL:
                    hidden.append((nx, ny))
                elif neighbor == FLAGGED_CELL:
                    flagged += 1

            remaining_mines = int(cell) - flagged
            if not hidden:
                continue
            if remaining_mines == 0:
                safe_cells.update(hidden)
            elif remaining_mines == len(hidden):
                mine_cells.update(hidden)

    return safe_cells, mine_cells


def score_minesweeper_move(prompt_or_state, completion, expected_completion=None):
    """Score one proposed move using format, legality, reference match, and visible-board logic."""
    move = parse_move(completion)
    if move is None:
        return -6.0

    reward = 0.5
    if move["action"] not in {"reveal", "flag"}:
        return -5.0

    board = parse_board(prompt_or_state)
    if board is not None:
        height = len(board)
        width = len(board[0]) if height else 0
        x, y = move["x"], move["y"]

        if y < 0 or y >= height or x < 0 or x >= width:
            return -25.0

        cell = board[y][x]
        safe_cells, mine_cells = infer_deterministic_cells(board)
        target = (x, y)

        if move["action"] == "reveal":
            if cell == HIDDEN_CELL:
                reward += 1.0
            elif cell == FLAGGED_CELL:
                reward -= 4.0
            else:
                reward -= 2.0

            if target in safe_cells:
                reward += 5.0
            if target in mine_cells:
                reward -= 10.0

        elif move["action"] == "flag":
            if cell == HIDDEN_CELL:
                reward += 0.5
            elif cell == FLAGGED_CELL:
                reward -= 1.0
            else:
                reward -= 3.0

            if target in mine_cells:
                reward += 5.0
            elif target in safe_cells:
                reward -= 5.0
            else:
                reward -= 1.0

    expected_move = parse_move(expected_completion) if expected_completion is not None else None
    if expected_move is not None:
        reward += 3.0 if move == expected_move else 0.0

    return float(reward)


def minesweeper_environment_reward(previous_state, next_state=None, invalid_move=False):
    """Reward from actual API/game feedback when a move has been executed."""
    if invalid_move:
        return -6.0
    if next_state is None:
        return 0.0

    status = next_state.get("status") if isinstance(next_state, dict) else None
    last_move = next_state.get("last_move") if isinstance(next_state, dict) else None
    if isinstance(last_move, dict) and "score_delta" in last_move:
        reward = float(last_move["score_delta"])
        if status == "won":
            reward += 75.0
    else:
        prev_score = previous_state.get("score", 0) if isinstance(previous_state, dict) else 0
        next_score = next_state.get("score", prev_score) if isinstance(next_state, dict) else prev_score
        reward = float(next_score - prev_score)

    if status == "lost":
        reward -= 50.0

    return reward


def minesweeper_reward_func(prompts=None, completions=None, answer=None, output=None, **kwargs):
    """TRL-compatible reward function.

    Use `answer` or `output` for dataset reference moves. When online game
    states are available, prefer `minesweeper_environment_reward` after each
    executed move because it can see true mines and terminal outcomes.
    """
    if completions is None:
        completions = kwargs.get("responses") or kwargs.get("generations") or []
    if prompts is None:
        prompts = kwargs.get("inputs") or kwargs.get("states") or [None] * len(completions)

    references = answer if answer is not None else output
    if references is None:
        references = kwargs.get("expected") or kwargs.get("labels")

    rewards = []
    for index, completion in enumerate(completions):
        prompt = prompts[index] if index < len(prompts) else None
        expected = references[index] if isinstance(references, (list, tuple)) and index < len(references) else references
        rewards.append(score_minesweeper_move(prompt, completion, expected))

    return rewards


#### Convert to HuggingFace compatible dataset

In [19]:
from datasets import Dataset
train_small_batch = train[0:300]
train_small_batch["text"] = tokenizer.apply_chat_template(train_small_batch["Messages"].values.tolist(), tokenize = False, add_generation_prompt = False)
train_small_batch = Dataset.from_pandas(train_small_batch)


In [20]:
del train
import gc
torch.cuda.empty_cache()
gc.collect()

2838

## Start The Fine-Tuning Process


In [21]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_small_batch,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 2,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [22]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
2,0.696000
4,0.434100
6,0.230000
8,0.203300
10,0.158900
12,0.215100
14,0.184100
16,0.158800
18,0.174100
20,0.182200


TrainOutput(global_step=300, training_loss=0.1177207089215517, metrics={'train_runtime': 1350.5137, 'train_samples_per_second': 0.444, 'train_steps_per_second': 0.222, 'total_flos': 9548204545167360.0, 'train_loss': 0.1177207089215517, 'epoch': 2.0})

In [33]:
test_message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test['input'][7]}
]

text = tokenizer.apply_chat_template(
    test_message,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)
text

'<|im_start|>system\n# Minesweeper AI System Prompt\n\nYou are an AI agent playing Minesweeper.\nYour objective is to maximize score while completing the game without revealing a mine.\n\n## Allowed Actions\n\nYou may send exactly one move at a time using one of these actions:\n\n- `reveal`\n- `flag`\n\nEach move must target exactly one tile coordinate:\n\n- `x`\n- `y`\n\nExample move payload:\n\n```json\n{\n  "action": "reveal",\n  "x": 3,\n  "y": 4\n}\n```\n\n## Rules You Must Follow\n\n- Revealing a flagged tile is invalid\n- Flagging a revealed tile is invalid\n- Revealing a mine ends the game immediately\n- The game is won only when all safe tiles are revealed and all mines are correctly flagged\n- Hidden mine locations are not exposed while the game is in progress\n\n## Scoring Rules\n\n- Reveal a safe tile: `+1` for each safe tile revealed\n- Correctly flag a mine: `+2`\n- Incorrectly flag a safe tile: `-2`\n- Reveal a mine: immediate loss\n- Win the full game: `+50`\n\n## State

In [35]:
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|im_start|>system
# Minesweeper AI System Prompt

You are an AI agent playing Minesweeper.
Your objective is to maximize score while completing the game without revealing a mine.

## Allowed Actions

You may send exactly one move at a time using one of these actions:

- `reveal`
- `flag`

Each move must target exactly one tile coordinate:

- `x`
- `y`

Example move payload:

```json
{
  "action": "reveal",
  "x": 3,
  "y": 4
}
```

## Rules You Must Follow

- Revealing a flagged tile is invalid
- Flagging a revealed tile is invalid
- Revealing a mine ends the game immediately
- The game is won only when all safe tiles are revealed and all mines are correctly flagged
- Hidden mine locations are not exposed while the game is in progress

## Scoring Rules

- Reveal a safe tile: `+1` for each safe tile revealed
- Correctly flag a mine: `+2`
- Incorrectly flag a safe tile: `-2`
- Reveal a mine: immediate loss
- Win the full game: `+50`

## State Interpretation

You receive game state that 

#### Save as .safetensors 16-bit

In [ ]:
import os
HF_USERNAME = os.getenv('HF_USERNAME')
HF_TOKEN = os.getenv('HF_TOKEN')

model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
model.push_to_hub_merged(f"{HF_USERNAME}/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = HF_TOKEN)



config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  603kB / 3.08GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:44<00:00, 142.02s/it]


Unsloth: Merge process complete. Saved to `/content/RealPirate786/qwen_finetune_16bit`


#### save as GGUF

In [ ]:
model.save_pretrained_gguf("qwen_finetune", tokenizer,)
model.push_to_hub_gguf(f"{HF_USERNAME}/qwen_finetune", tokenizer, token = HF_TOKEN, quantization_method = 'q8_0')


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-instruct-2507.Q8_0.gguf:   1%|          | 24.0MB / 4.28GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/RealPirate786/qwen_finetune
Unsloth: Cleaning up temporary files...


'RealPirate786/qwen_finetune'